# Event Impact Analysis

Effect of public holidays, events and event size on `arrival_delay`.

## Setup

In [ ]:
import polars as pl
from pathlib import Path

from wgnd.core.theme import setup
from wgnd.core._output import section_header, log, success

from zh_tram_flow.config import PATHS
from zh_tram_flow.settings import setup_plotting, logger

setup_plotting()
setup()
logger.info("notebook started")

TRAIN = PATHS["processed"] / "train_features.parquet"
TEST  = PATHS["processed"] / "test_features.parquet"

lf = pl.scan_parquet(TRAIN)

## Holidays

Public holidays vs normal days — does reduced traffic improve or worsen punctuality?

In [ ]:
section_header("Holiday Impact")

holidays = (
    lf.group_by("is_holiday")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("is_holiday")
    .collect()
)

## Event Days vs Normal Days

`has_event = True` vs `False` — measurable crowd effect on tram network?

In [ ]:
section_header("Event Days vs Normal Days")

event_days = (
    lf.group_by("has_event")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("has_event")
    .collect()
)

## Event Size

Does a larger event (weight 1→3) cause proportionally more delay?

In [ ]:
section_header("Event Size Impact")

event_size = (
    lf.group_by("event_weight")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("event_weight")
    .collect()
)

## Event Type

Delay breakdown by event category: Feiertag, Stadtfest, Konzert, Messe, Fussball.

In [ ]:
section_header("Event Type Breakdown")

event_type = (
    lf.filter(pl.col("has_event"))
    .group_by("event_type")
    .agg(pl.col("arrival_delay").mean().alias("avg_delay"), pl.len().alias("count"))
    .sort("avg_delay", descending=True)
    .collect()
)